# 3D Engine — TRELLIS.2 → GLB

**Independent engine:** reference image → PBR 3D asset.

Use this for objects, props, furniture, buildings **and** characters. Nothing is sent to the Animation Engine automatically.

This notebook runs **TRELLIS.2 directly** (no ComfyUI) and is pinned to a tested upstream commit so a future upstream API change does not silently break the notebook.

**Recommended runtime:** Colab A100 / NVIDIA GPU with at least 24 GB VRAM.

In [ ]:
# Preflight: fail early instead of downloading many GB on the wrong runtime.
import shutil, subprocess, pathlib, os, re

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=True
).stdout.strip().splitlines()
if not smi:
    raise RuntimeError("No NVIDIA GPU detected. In Colab choose Runtime → Change runtime type → GPU.")

gpu_name, memory_mib = [x.strip() for x in smi[0].rsplit(",", 1)]
memory_mib = int(memory_mib)
free_gib = shutil.disk_usage("/content").free / (1024**3)
print(f"GPU: {gpu_name} | VRAM: {memory_mib/1024:.1f} GiB | Free disk: {free_gib:.1f} GiB")

if memory_mib < 24000:
    raise RuntimeError(
        "This notebook intentionally follows the official TRELLIS.2 >=24 GB path. "
        "Switch the Colab runtime to A100/H100 instead of using low-VRAM hacks."
    )
if free_gib < 35:
    raise RuntimeError("Less than 35 GiB free disk space; restart with a fresh Colab runtime.")


In [ ]:
# Get only our small engine helpers.
import pathlib, shutil

if pathlib.Path("/content/My-works").exists():
    shutil.rmtree("/content/My-works")
!git clone -q --depth 1 https://github.com/Logan17de/My-works.git /content/My-works

TOOLS = "/content/My-works/ai-3d-animation-engines/3d-engine"
print("Helpers:", TOOLS)


## Install pinned TRELLIS.2

The runtime is disposable. The notebook recreates the environment cleanly on every install-cell run.

TRELLIS.2's upstream setup pins PyTorch 2.6 + CUDA 12.4. If Colab does not expose a system CUDA 12.4 toolkit, this cell installs CUDA Toolkit 12.4 **inside the Conda environment** so the CUDA extensions compile against the matching toolkit.

In [ ]:
%%bash
set -euo pipefail

TRELLIS_REF="75fbf0183001ed9876c8dbb35de6b68552ee08bd"

apt-get update -qq
apt-get install -y -qq git git-lfs build-essential cmake ninja-build wget ffmpeg sudo \
    libjpeg-dev libgl1-mesa-dev libegl1-mesa-dev

if [ ! -x /opt/conda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /opt/conda
fi
source /opt/conda/etc/profile.d/conda.sh

rm -rf /content/TRELLIS.2 /tmp/extensions
git clone -q --recursive https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2
git -C /content/TRELLIS.2 checkout -q "$TRELLIS_REF"
git -C /content/TRELLIS.2 submodule sync --recursive
git -C /content/TRELLIS.2 submodule update --init --recursive --force

conda env remove -n trellis2 -y >/dev/null 2>&1 || true
conda create -n trellis2 python=3.10 -y -q
conda activate trellis2

python -m pip install -q --upgrade pip setuptools wheel packaging ninja
python -m pip install torch==2.6.0 torchvision==0.21.0 \
    --index-url https://download.pytorch.org/whl/cu124

if [ -x /usr/local/cuda-12.4/bin/nvcc ]; then
  export CUDA_HOME=/usr/local/cuda-12.4
else
  echo "System CUDA 12.4 toolkit not found; installing matching toolkit in Conda env..."
  conda install -y -q -c nvidia/label/cuda-12.4.1 cuda-toolkit
  export CUDA_HOME="$CONDA_PREFIX"
fi
export PATH="$CUDA_HOME/bin:$PATH"
echo "CUDA_HOME=$CUDA_HOME"
"$CUDA_HOME/bin/nvcc" --version | tail -n 1

cd /content/TRELLIS.2
# Environment already exists, so deliberately omit --new-env.
. ./setup.sh --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm

python - <<'PY'
import torch, o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline
print("TRELLIS.2 import smoke test: OK")
print("PyTorch:", torch.__version__, "| CUDA runtime:", torch.version.cuda)
assert torch.cuda.is_available()
PY


In [ ]:
# Upload exactly one reference image.
from google.colab import files
import pathlib

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one image.")

input_name = next(iter(uploaded))
INPUT_IMAGE = f"/content/{input_name}"
if pathlib.Path(INPUT_IMAGE).suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp", ".bmp"}:
    raise ValueError("Expected PNG/JPG/JPEG/WEBP/BMP.")
print("Input:", INPUT_IMAGE)


In [ ]:
# Generate the PBR GLB. The helper exports the GLB BEFORE attempting preview rendering,
# so a preview problem cannot throw away a successful 3D generation.
import subprocess, pathlib, shlex

OUTPUT_DIR = "/content/trellis_outputs"
ASSET_NAME = "asset"

cmd = [
    "/opt/conda/bin/conda", "run", "-n", "trellis2",
    "python", f"{TOOLS}/run_trellis2.py",
    "--input", INPUT_IMAGE,
    "--output-dir", OUTPUT_DIR,
    "--name", ASSET_NAME,
    "--envmap", "/content/TRELLIS.2/assets/hdri/forest.exr",
    "--decimation-target", "1000000",
    "--texture-size", "4096",
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd="/content/TRELLIS.2", check=True)

GLB_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}.glb"
PREVIEW_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_preview.mp4"
if not pathlib.Path(GLB_PATH).is_file():
    raise RuntimeError("TRELLIS.2 finished without creating the GLB.")
print(f"GLB: {GLB_PATH} ({pathlib.Path(GLB_PATH).stat().st_size/1024**2:.1f} MiB)")


In [ ]:
# Preview if the optional render succeeded.
from IPython.display import Video, display
import pathlib

if pathlib.Path(PREVIEW_PATH).is_file():
    display(Video(PREVIEW_PATH, embed=True))
else:
    print("No MP4 preview was produced. The GLB itself is still valid.")


In [ ]:
# Download the generated asset.
from google.colab import files
files.download(GLB_PATH)


## Optional — save outputs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/AI-3D-Engine"
# !cp -f /content/trellis_outputs/* "/content/drive/MyDrive/AI-3D-Engine/"
